## A Naive RAG Pipeline

The diagram below shows the three broad stages of a **Naive RAG Pipeline**: _Indexing_, _Retrieval_, and _Generation_. This notebook walks through all three and builds a working pipeline end-to-end.

![RAG Pipeline](images/rag_pipeline.png)

---

### Stage 1 — Indexing

Indexing is a one-time preparation step. You process your documents ahead of time and store them in a form that can be searched quickly at query time. It has four sub-steps:

1. **Loading** — Read your source documents (PDFs, web pages, text files, etc.) into memory. Each document becomes a `Document` object that carries both the raw text and metadata such as the source file name and page number.

2. **Splitting** — Large documents are too big to fit into most LLM's context window in one go. So you break them into smaller, overlapping _chunks_ — typically `~1000` characters each, with `~200` characters of overlap between adjacent chunks. The overlap prevents a sentence that straddles two chunks from being lost entirely.

3. **Embedding** — Each chunk is converted into a _vector_ (a list of numbers) by an _embedding model_. Vectors capture the semantic meaning of the text. Chunks that _talk_ about similar things end up close together in vector space, even if they use different words. For OpenAI-based pipelines we use `OpenAIEmbeddings()`.

4. **Storing** — The vectors (and their corresponding text chunks) are saved into a _vector store_ such as [Chroma](https://www.trychroma.com/). This is optional for a quick demo — you can keep everything in memory — but persisting to disk means you only pay the embedding cost once.

---

### Stage 2 — Retrieval

When a user asks a question, the pipeline needs to find the chunks most likely to contain the answer:

1. The user's question is embedded using the **same embedding model** used during indexing, producing a query vector.
2. The vector store compares the query vector against all stored chunk vectors and returns the `k` closest matches (by default `k = 4`). These are the chunks most semantically similar to the question asked by the user.

---

### Stage 3 — Generation

With the relevant chunks in hand, the pipeline constructs an answer:

1. The retrieved chunks are concatenated into a single block of text called the _context_.
2. A prompt is built that contains both the _context_ and the user's _question_, and is sent to the LLM.
3. The LLM reads the context and generates an answer grounded in it, rather than relying solely on its training data.

This is the key idea behind RAG: instead of asking the model to remember facts, you _retrieve_ the facts at query time and _hand them to the model_ as "reference material" (context) from which the LLM generates its response.

## 01 Single Document RAG Pipeline
In this section we'll illustrate the above Naive RAG pipeline on a single PDF file. We'll be using an OpenAI LLM (specifically, `gpt-5-nano`), OpenAI Embeddings and the Chroma vector store. 


In [2]:
from dotenv import load_dotenv
from rich.console import Console

import langchain

from langsmith import Client as LangSmithClient
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

print(f"Using Langchain {langchain.__version__}")

Using Langchain 1.2.14


In [3]:
load_dotenv(override=True)
console = Console()

### Step 01 - Indexing

For this example, we'll be indexing Nvidia's **FY2025 Annual Report (Form 10-K)** — a 100+ page SEC filing that covers Nvidia's financial results, business segments, GPU product lines, risk factors, and strategic outlook for fiscal year 2025. It's a strong choice for testing a RAG pipeline because:

- It is a **post-training document** — filed with the SEC in early 2026, it lies beyond GPT-5-nano's training cutoff, so the model cannot fall back on memorised facts and is forced to ground every answer in the retrieved context.
- It is **long and multi-sectional** — financial statements, segment revenues, GPU product descriptions, risk factors, and legal disclosures each live in distinct sections, making cross-section retrieval genuinely hard.
- It is **dense with specific, verifiable facts** — exact revenue figures per business segment, named GPU product lines (H100, B200, etc.), precise headcount, capital expenditure numbers, and quantified risk disclosures that require precise retrieval to answer correctly.

Indexing follows 3 steps:

1. **Loading** — we use LangChain's `PyPDFLoader` to parse the local PDF file page by page into a list of `Document` objects.
2. **Splitting** — we use `RecursiveCharacterTextSplitter` to break each page into overlapping chunks of ~1000 characters. Overlapping ensures that sentences spanning page or chunk boundaries are not silently cut off.
3. **Embedding & Storing** — each chunk is embedded using OpenAI embeddings and stored in a **Chroma** vector store, which will serve as our retriever.

In [5]:
# Step 1 - Load the PDF (must be in the same folder as this notebook)
loader = PyPDFLoader("Nvidia-10k-Feb-26-2025.pdf")
docs = loader.load()
print(f"Loaded {len(docs)} pages from PDF")

# Step 2 - Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
print(f"Split into {len(splits)} chunks")

# Step 3 - Embed the vectors above
# in this example we WON'T be saving the vector store to an
# offline database. But if you wanted to, just uncomment the
# commented line below
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=OpenAIEmbeddings(),
    # uncomment this line to store the vector store
    # to an offline database
    # persist_directory="chroma_db/Nvidia10k",
)

retriever = vectorstore.as_retriever()

Loaded 130 pages from PDF
Split into 688 chunks


In [7]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""You are an assistant for 
question-answering tasks. Use the following pieces of retrieved context 
to answer the question. If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.
\nQuestion: {question} 
\nContext: {context}
\nAnswer:""")

In [8]:
# create my LLM - we'll be using GPT-5-nano
llm = ChatOpenAI(model_name="gpt-5-nano", temperature=0)

In [9]:
# Post-processing - separate each doc in the context with 2 blank lines
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# Chain
rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

Here are some questions you can ask off the vector store, which should work:

1. What was Nvidia's total revenue for fiscal year 2025?
2. What are the primary end markets Nvidia's Data Center segment serves?
3. How many full-time employees did Nvidia have at the end of fiscal year 2025?
4. What GPU products does Nvidia list under its Compute & Networking segment?
5. What does Nvidia describe as its primary competition in the Gaming segment?
6. Where are Nvidia's principal offices located?

And here are some questions that _should NOT_ work:
1. How did the revenue growth in the Data Center segment compare to the increase in R&D headcount, and what product launches drove both?
2. What specific supply chain risks does Nvidia cite, and how do those risks map to the geographic revenue concentration shown in the financial statements?
3. How does Nvidia's stated AI strategy in the business overview align with the capital expenditure figures reported in the cash flow statement?
4. Which risk factors relate to export controls, and how do those restrictions affect the revenue contribution of Nvidia's top three international markets?
5. What are Nvidia's stated competitive advantages in the professional visualisation market, and how do the segment's operating margins reflect those advantages?
6. How does the increase in stock-based compensation expense correlate with headcount growth across segments, and what does management say about retention strategy?

These fail because the answer requires the retriever to surface chunks from two or more distant sections simultaneously — financials + strategy, risk factors + geographic revenue, etc. The retriever will fetch chunks biased toward one half of the query and the LLM fills the gap with plausible-sounding fabrication, exactly the failure mode the notebook is demonstrating.

In [11]:
# Simple, focused question - naive RAG handles this fine
rag_chain.invoke("What was Nvidia's total revenue for fiscal year 2025?")

"Nvidia's total revenue for fiscal year 2025 was $130.5 billion (about $130,497 million)."

**How does this all work?**

* LangChain provides a lot of classes, such as retrievers, ChatPromptTemplate, your LLMs, parsers (such as StrOutputParser). 
* Each of these objects are derived from a `Runnable` class - this is the secret sauce! 
* The `Runnable` class provides an `invoke()` method, so all `Runnable`s can be `invoke()`-ed! 
* Further, all `Runnables` can be chained together using the `|` operator, giving you a chain of execution, which is also a `Runnable`! 

> **Wait a minute!** 💡
> 
> But `{"context": retriever | RunnableLambda(format_docs), "question": RunnablePassthrough()}` is a plain Python `dict` — and a `dict` is _NOT_ a `Runnable`!<br/>
> So how can we use `|` to pipe into it? How does this even work?? Following line should fail! 🤔
> ```python
> rag_chain = {"context": retriever | RunnableLambda(format_docs), "question": RunnablePassthrough()} | prompt | ...
> ```

Great observation 👀! The secret is Python's **operator protocol** and LangChain's `__ror__` (reverse `or`) implementation.

When Python sees `dict | prompt`, it tries two things in order:
* **First** — it calls `dict.__or__(prompt)`. A plain Python `dict` has no idea what a `Runnable` is, so it returns `NotImplemented`.
* **Fallback** — Python then calls `prompt.__ror__(dict)`. LangChain's `Runnable` base class _does_ implement `__ror__`, and it quietly **wraps that dict into a `RunnableParallel`** on the fly!

So the `dict` never becomes a `Runnable` itself — instead, `prompt.__ror__` converts the whole expression into a `RunnableParallel` that evaluates _each value in the dict in parallel_, then passes the assembled result forward to `prompt`. That's the real magic! 🤩

---

**So what _exactly_ is `RunnableLambda(...)`?** 🤔

Think of your LangChain chain as a factory assembly line 🏭. Every workstation on the line must speak the same protocol — the `Runnable` interface (i.e., expose an `invoke()` method). Your own plain Python function `format_docs` doesn't speak that protocol — it's just a regular Python function!

`RunnableLambda` is the **adapter plug** 🔌 that wraps your function and makes it assembly-line compatible (i.e. converts a regular Python function into a `Runnable`💡):

```python
# format_docs is just a plain Python function — you cannot chain it with |
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Wrap it in RunnableLambda and now it IS a Runnable — chain away! 🎉
retriever | RunnableLambda(format_docs)
```

Without `RunnableLambda`, writing `retriever | format_docs` raises a `TypeError` — `format_docs` has no `invoke()` method, so `|` doesn't know what to do with it. Wrap it, and it just works! 🤩

---

**And `RunnablePassthrough()` — what's the point of that?** 🤔

Imagine the chain as a series of pipes. Each pipe _transforms_ the data flowing through it. But sometimes you don't want any transformation at all — you simply want the original input to flow through unchanged so it can be used as a variable downstream.

That's exactly what `RunnablePassthrough` does — it's a **transparent pipe** 🪟 that lets the input pass through completely untouched.

Here is the full picture of what happens when you call `rag_chain.invoke("What is Task Decomposition?")`:

```python
{
    "context": retriever | RunnableLambda(format_docs),  # input → retriever → format_docs → context string
    "question": RunnablePassthrough(),                    # input → unchanged → "What is Task Decomposition?"
}
```

* The `"context"` branch **transforms** the input: it finds the relevant chunks via `retriever`, then joins them into a single formatted string via `format_docs`. That string _is_ the value of `{context}` in your prompt.
* The `"question"` branch does **nothing** to the input: the original question string flows through as-is and becomes the value of `{question}` in your prompt.

Both branches run in parallel (that's the `RunnableParallel` magic from above!), and the assembled `{"context": "...", "question": "..."}` dict is then fed into `prompt`. 🤩

In [12]:
# Compound query spanning two distant sections - watch naive RAG struggle
rag_chain.invoke(
    """How did the revenue growth in the Data Center segment compare to the increase in R&D headcount, and what product launches drove both?"""
)

'- Data Center revenue rose 142% year over year (computing up 162% driven by Hopper; networking up 51%).\n\n- The launches driving this growth were the Hopper accelerated computing platform and the production launch of the Blackwell architecture.\n\n- The provided context does not include the R&D headcount increase, so I can’t compare that to the revenue growth.'

In [13]:
# Another cross-section compound query - naive RAG retrieves the wrong chunks
rag_chain.invoke(
    """What are Nvidia's stated competitive advantages in the professional visualisation market, and how do the segment's operating margins reflect those advantages?"""
)

'NVIDIA’s competitive advantages in Professional Visualization come from its accelerated computing platform and full-stack innovation (across architecture, chip design, systems, interconnect, algorithms, and software), delivering much faster problem-solving with lower power consumption and order-of-magnitude performance gains over legacy approaches. The provided text does not include the Graphics/Professional Visualization segment’s operating margins. Therefore, I can’t say how margins reflect these advantages from the given information.'

Do you see a problem with the responses to the above queries? Probably not, unless you read the PDF thoroughly! While  the model is generating plausible-sounding but unreliable answers, not genuinely answering from retrieved context. 

Look closely:
* **The benchmark query**: No specific benchmark names or scores cited — just vague generalities about "cybersecurity" and "chemical/biological weapons." A real answer from the paper would name exact test sets with numbers.
* **The attention heads query**: 128 attention heads → outperforms on MGSM" — the causal link between those two facts is fabricated. The paper never makes that argument. The model stitched together two unrelated retrieved fragments and invented a narrative connecting them.

In [14]:
# What did the retriever actually fetch for this compound query?
question = """How did the revenue growth in the Data Center segment compare to the increase in R&D headcount, and what product launches drove both?"""

for i, doc in enumerate(retriever.invoke(question)):
    print(f"--- Chunk {i+1} (page {doc.metadata.get('page', '?')}) ---")
    print(doc.page_content[:400])
    print()

--- Chunk 1 (page 40) ---
Table of Contents
Reportable Segments
Revenue by Reportable Segments
Year Ended
Jan 26, 2025 Jan 28, 2024 $Change %Change
($ in millions)
Compute & Networking $ 116,193 $ 47,405 $ 68,788 145 %
Graphics 14,304 13,517 787 6 %
Total $ 130,497 $ 60,922 $ 69,575 114 %
Operating Income by Reportable Segments
Year Ended
Jan 26, 2025 Jan 28, 2024 $Change %Change
($ in millions)
Compute & Networking $ 82,8

--- Chunk 2 (page 40) ---
Table of Contents
Reportable Segments
Revenue by Reportable Segments
Year Ended
Jan 26, 2025 Jan 28, 2024 $Change %Change
($ in millions)
Compute & Networking $ 116,193 $ 47,405 $ 68,788 145 %
Graphics 14,304 13,517 787 6 %
Total $ 130,497 $ 60,922 $ 69,575 114 %
Operating Income by Reportable Segments
Year Ended
Jan 26, 2025 Jan 28, 2024 $Change %Change
($ in millions)
Compute & Networking $ 82,8

--- Chunk 3 (page 37) ---
Data Center revenue for fiscal year 2025 was up 142% from a year ago. The strong year-on-year growth was driven by dem

You'll see that the 4 retrieved chunks address one half of the query at best — and the LLM filled the gap with confident-sounding fabrication. That's the real failure of naive RAG, and it's a much more honest and compelling segue into query translation than a broken answer.

These are not complete responses from the LLM. Why is that happening? We'll answer that in the next notebook in the series.